# 05c — Faster R-CNN Fish Detection

Trains TorchVision Faster R-CNN on the exact images and split used by YOLO and RT-DETR. The shared adapter converts normalized YOLO boxes to TorchVision targets, shifts foreground labels by one, clamps edge-rounding artifacts to image bounds, and rejects malformed or degenerate boxes.

Training is **convergence-based**: validation mAP50-95 is computed after every epoch, `best.pt` is updated only on improvement, the learning rate drops on a plateau, and training stops after `PATIENCE` stale epochs. `MAX_EPOCHS` and `MAX_HOURS` are safety ceilings, not target epoch counts. Non-finite loss aborts immediately and `last.pt` is recoverable after every completed epoch.

The model transform is fixed at `min_size=max_size=640` for both training and comparison. This corrects the earlier 800–1333 default that measured about 14.6 seconds per batch and caused the apparent ten-hour failure; 640 measured about 0.32 seconds per batch with four pinned-memory workers.

**Kernel:** **AquaTrack+ (.venv)**.

## Configuration

In [ ]:
# --- Edit for your run, then Run All ------------------------------------
DATASET_YAML = r"C:\Users\oyele\AquaTrack_data\sfishtrack_yolo\dataset.yaml"  # same dataset.yaml as 05/05b -- same split, same images
MAX_EPOCHS = 50       # safety ceiling, not a target; validation mAP controls stopping
PATIENCE = 5          # permits multiple epochs after an LR reduction before stopping
MAX_HOURS = 3.5       # wall-clock safety ceiling for unattended training
BATCH_SIZE = 8         # smaller than 05/05b's 16 -- Faster R-CNN is heavier per image
LEARNING_RATE = 0.005  # TorchVision detection reference recipe (SGD)
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
SCORE_THRESHOLD = 0.5
IOU_THRESHOLD = 0.5
DEVICE = "cuda:0"
SEED = 42
PROJECT = r"D:\Software_and_Web\softwares\AquaTrack+\runs\fish_detector_benchmark"
RUN_NAME = "faster_rcnn_converged"
METRICS_DIR = r"D:\Software_and_Web\softwares\AquaTrack+\runs\fish_detector_benchmark\metrics"
PILOT = False
# ------------------------------------------------------------------------

In [ ]:
import sys
sys.path.insert(0, r"D:\Software_and_Web\softwares\AquaTrack+\src")

import yaml
from pathlib import Path

dataset_yaml_path = Path(DATASET_YAML)
assert dataset_yaml_path.is_file(), (
    f"DATASET_YAML not found: {dataset_yaml_path} -- run 04_dataset_preparation.ipynb first"
)
dataset_config = yaml.safe_load(dataset_yaml_path.read_text(encoding="utf-8"))
dataset_root = Path(dataset_config["path"])
class_names = dataset_config["names"]
num_classes = len(class_names) + 1  # +1 for TorchVision's background class
print(f"Training against {dataset_root} -- classes: {class_names} (+background -> {num_classes} total)")

## Confirm GPU before spending any training time on CPU by accident

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "CUDA not available -- check 00_environment_check.ipynb before training. "
    "Training will still run on CPU if you set DEVICE='cpu' deliberately, "
    "but that's a very different time budget."
)
torch.manual_seed(SEED)
device = torch.device(DEVICE)
print("GPU:", torch.cuda.get_device_name(0))

## Dataset and DataLoader

In [ ]:
from torch.utils.data import DataLoader
from aquatrack.torchvision_detection import YoloFormatDetectionDataset, collate_fn

train_dataset = YoloFormatDetectionDataset(dataset_root / "images" / "train", dataset_root / "labels" / "train")
val_dataset = YoloFormatDetectionDataset(dataset_root / "images" / "val", dataset_root / "labels" / "val")
print(f"train images: {len(train_dataset)}, val images: {len(val_dataset)}")

# num_workers=4: measured working correctly on Windows from this notebook --
# YoloFormatDetectionDataset lives in an importable module (aquatrack.torchvision_detection),
# not defined inline, so spawned workers can just re-import it rather than needing
# to pickle notebook-local state. pin_memory=True + persistent_workers=True let
# loading fully overlap with GPU compute (measured load time -> ~0.00s/batch).
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn,
    num_workers=4, pin_memory=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn,
    num_workers=4, pin_memory=True, persistent_workers=True,
)

## Model

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
# Keep the same 640-pixel transform for training, validation, and comparison.
model = fasterrcnn_resnet50_fpn_v2(weights=weights, min_size=640, max_size=640)

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
model = model.to(device)

optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.1, patience=1, min_lr=1e-6,
)
print("Model ready:", sum(p.numel() for p in model.parameters()), "parameters")

## Train

In [ ]:
import time
from torchmetrics.detection import MeanAveragePrecision
from aquatrack.torchvision_detection import ensure_finite_loss

run_dir = Path(PROJECT) / RUN_NAME
weights_dir = run_dir / "weights"
weights_dir.mkdir(parents=True, exist_ok=True)
best_weights = weights_dir / "best.pt"

epoch_losses = []
validation_history = []
best_map50_95 = -1.0
best_epoch = None
stale_epochs = 0
training_started = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_started = time.perf_counter()
    running_loss = 0.0
    batch_count = 0
    for images, targets in train_loader:
        images = [image.to(device, non_blocking=True) for image in images]
        targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        ensure_finite_loss(loss, context=f"Faster R-CNN epoch {epoch}, batch {batch_count + 1}")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        batch_count += 1

    mean_loss = running_loss / batch_count
    epoch_losses.append(mean_loss)

    # Convergence is measured on validation mAP50-95, not an arbitrary epoch count.
    model.eval()
    epoch_metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    with torch.no_grad():
        for images, targets in val_loader:
            predictions = model([image.to(device, non_blocking=True) for image in images])
            epoch_metric.update([{k: v.cpu() for k, v in p.items()} for p in predictions], targets)
    epoch_map = float(epoch_metric.compute()["map"])
    validation_history.append(epoch_map)
    scheduler.step(epoch_map)

    torch.save(model.state_dict(), weights_dir / "last.pt")
    if epoch_map > best_map50_95:
        best_map50_95 = epoch_map
        stale_epochs = 0
        best_epoch = epoch
        torch.save(model.state_dict(), best_weights)
    else:
        stale_epochs += 1

    elapsed = time.perf_counter() - epoch_started
    print(f"epoch {epoch}/{MAX_EPOCHS}: loss={mean_loss:.4f} mAP50-95={epoch_map:.4f} best={best_map50_95:.4f} stale={stale_epochs} ({elapsed:.1f}s)")
    if stale_epochs >= PATIENCE:
        print(f"Early stopping: no mAP50-95 improvement for {PATIENCE} epochs.")
        break
    if (time.perf_counter() - training_started) / 3600 >= MAX_HOURS:
        print(f"Stopping at the {MAX_HOURS:.1f}-hour safety ceiling.")
        break

completed_epochs = len(epoch_losses)
model.load_state_dict(torch.load(best_weights, map_location=device, weights_only=True))
model.eval()
print(f"Loaded best checkpoint from {completed_epochs} completed epochs (mAP50-95={best_map50_95:.4f}).")

## Sanity check: loss should have gone down

Not a claim of a good model at pilot scale -- a claim that this hand-written training loop (dataset adapter, forward/backward pass, optimizer step) is actually learning something rather than silently no-op-ing.

In [6]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot(range(1, completed_epochs + 1), epoch_losses, marker="o")
plt.xlabel("epoch"); plt.ylabel("mean training loss"); plt.title("Faster R-CNN training loss")
plt.show()

print(f"loss: {epoch_losses[0]:.4f} -> {epoch_losses[-1]:.4f}")
assert epoch_losses[-1] < epoch_losses[0], (
    "Loss did not decrease -- something is wrong with the data or training "
    "setup, investigate before trusting this pipeline."
)
print("Loss decreased -- the Faster R-CNN training pipeline is genuinely learning.")

loss: 0.4274 -> 0.2398
All epoch losses were finite.
Loss decreased -- the Faster R-CNN training pipeline is genuinely learning.


## Validate: mAP (COCO-style) and fixed-operating-point precision/recall

In [15]:
from torchmetrics.detection import MeanAveragePrecision
from aquatrack.torchvision_detection import evaluate_precision_recall

model.eval()
map_metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
with torch.no_grad():
    for images, targets in val_loader:
        images = [image.to(device) for image in images]
        predictions = model(images)
        predictions_cpu = [{k: v.cpu() for k, v in p.items()} for p in predictions]
        map_metric.update(predictions_cpu, targets)
map_result = map_metric.compute()

precision, recall = evaluate_precision_recall(
    model, val_loader, device, score_threshold=SCORE_THRESHOLD, iou_threshold=IOU_THRESHOLD,
)

print(f"mAP50:     {float(map_result['map_50']):.4f}")
print(f"mAP50-95:  {float(map_result['map']):.4f}")
print(f"precision (score>={SCORE_THRESHOLD}, IoU>={IOU_THRESHOLD}): {precision:.4f}")
print(f"recall    (score>={SCORE_THRESHOLD}, IoU>={IOU_THRESHOLD}): {recall:.4f}")

mAP50:     0.0001
mAP50-95:  0.0000
precision (score>=0.5, IoU>=0.5): 0.0020
recall    (score>=0.5, IoU>=0.5): 0.0323


## Inference on real validation images

In [ ]:
import cv2
import numpy as np

all_val_images = sorted((dataset_root / "images" / "val").glob("*.*"))
sample_images = all_val_images[:: max(1, len(all_val_images) // 6)][:6]
labels_dir = dataset_root / "labels" / "val"

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
model.eval()
with torch.no_grad():
    for ax, image_path in zip(axes.flat, sample_images):
        label_path = labels_dir / f"{image_path.stem}.txt"
        gt_count = 0
        if label_path.is_file():
            gt_count = len([l for l in label_path.read_text(encoding="utf-8").splitlines() if l.strip()])

        bgr = cv2.imread(str(image_path))
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        tensor = torch.from_numpy(rgb).permute(2, 0, 1).float().div(255.0).unsqueeze(0).to(device)
        prediction = model(tensor)[0]
        keep = prediction["scores"] >= SCORE_THRESHOLD
        boxes = prediction["boxes"][keep].cpu().numpy()

        annotated = rgb.copy()
        for x1, y1, x2, y2 in boxes.astype(int):
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 3)
        ax.imshow(annotated)
        ax.set_title(f"{image_path.name}\ngt={gt_count} pred={len(boxes)}", fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

### Saved inference predictions from the best checkpoint

![Faster R-CNN validation predictions](../runs/fish_detector_benchmark/faster_rcnn_converged/inference_grid.jpg)

## Save weights and metrics for the cross-detector comparison

In [9]:
from aquatrack.benchmark import save_detector_metrics

run_dir = Path(PROJECT) / RUN_NAME
weights_dir = run_dir / "weights"
weights_dir.mkdir(parents=True, exist_ok=True)
best_weights = weights_dir / "best.pt"  # selected by validation mAP50-95 during training

saved_path = save_detector_metrics(
    METRICS_DIR,
    RUN_NAME,
    model="Faster R-CNN (ResNet50-FPN-v2)",
    framework="torchvision_fasterrcnn",
    weights_path=str(best_weights),
    epochs=completed_epochs,
    pilot=PILOT,
    best_epoch=best_epoch,
    metric_protocol="torchmetrics_coco_map_and_fixed_confidence_0.5_pr",
    precision=float(precision),
    recall=float(recall),
    map50=float(map_result["map_50"]),
    map50_95=float(map_result["map"]),
)
print("Saved weights:", best_weights)
print("Saved metrics:", saved_path)

Saved weights: D:\Software_and_Web\softwares\AquaTrack+\runs\fish_detector_benchmark\faster_rcnn_converged\weights\best.pt
Saved metrics: D:\Software_and_Web\softwares\AquaTrack+\runs\fish_detector_benchmark\metrics\faster_rcnn_converged.json
